<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 28px; border-radius: 24px; text-align: center; color: #f2f5f7; box-shadow: 0 8px 20px rgba(0,0,0,0.12);">
  <h1 style="font-size: 42px; margin-bottom: 8px;">🔧 Classical Restoration 🔧</h1>
  <h2 style="font-size: 24px; margin-top: 0;">01 · Part A — DSP Pipeline (OpenCV, Partly Hand-Implemented)</h2>
  <p style="font-size: 18px;">Tuned on LoLI-Street val, frozen once, applied unchanged to ExDark.</p>
</div>

### Imports

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
from classical import clahe_gamma, denoise_frequency, denoise_spatial, morphology, retinex
from classical.pipeline import CANDIDATES, FROZEN, restore
from metrics.psnr_ssim import all_metrics

VAL = REPO / "LoLI-Street: Low-Light Image Enhancement of Street" / "LoLI-Street Dataset" / "Val"
SAMPLE = sorted((VAL / "low").glob("*.jpg"))[0]
print("sample:", SAMPLE.name, "| frozen pipeline:", FROZEN)

<div style="background: linear-gradient(135deg, #74c69d, #48cae4); padding: 26px; border-radius: 22px; margin-top: 28px; border: 4px solid #2d6a4f; box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18); color: #081c15;">
  <h1 style="color: #081c15; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">💡 Step 1: Brightening on the luminance channel</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #133f30; font-weight: 500;">
    Global equalization, CLAHE, and per-image adaptive gamma — all on Y (YCrCb), never per-RGB,
    so brightness changes without hue shifts. 5-sample PSNR: equalizeHist 22.7, gamma 15.5, CLAHE 9.6, raw 6.8.
  </p>
</div>

### Brightening demo

In [ ]:
low = cv2.imread(str(SAMPLE))
high = cv2.imread(str(VAL / "high" / SAMPLE.name))
outs = {"low": low, "equalizeHist": clahe_gamma.equalize_hist_luminance(low),
        "clahe": clahe_gamma.apply_clahe(low), "gamma": clahe_gamma.adaptive_gamma(low), "high": high}
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
for ax, (k, v) in zip(axes, outs.items()):
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(k)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(135deg, #e8b7e8, #ffb6f4); padding: 24px; border-radius: 22px; margin-top: 28px; border: 4px solid #481344; color: #2b0a2a;">
  <h1 style="color: #2b0a2a;">🌓 Step 2: Single-Scale Retinex (hand-implemented)</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #2b0a2a;">
    log R = log I − log Gaussian(I, σ): illumination divided out in the log domain.
    5-sample PSNR ≈ 12.3 (σ=80) — below global equalization on these synthetic exposure pairs,
    kept as a candidate because it normalizes illumination instead of matching exposure.
  </p>
</div>

### Retinex demo

In [ ]:
ssr = retinex.single_scale_retinex(low)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (k, v) in zip(axes, [("low", low), ("ssr-σ80", ssr), ("high", high)]):
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(k)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(135deg, #9668b1, #8fb5d7, #78b9f6); padding: 30px; border-radius: 26px; color: #1d3557; box-shadow: 0 8px 22px rgba(0,0,0,0.16); margin-top: 28px;">
  <h1 style="color: #1d3557; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">🔇 Steps 3-4: Denoising — spatial vs frequency</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #1d3557;">
    Mean, median, Gaussian, bilateral vs FFT Gaussian low-pass, scored by PSNR/SSIM and
    Sobel-edge MSE. 5-sample winner: frequency σ=0.25 (PSNR 15.45, SSIM 0.626, edgeMSE 11174),
    ahead of bilateral on edges (13825) and Gaussian on SSIM (0.573).
  </p>
</div>

### Denoising demo

In [ ]:
bright = clahe_gamma.adaptive_gamma(low)
outs = {"bright": bright, "bilateral": denoise_spatial.denoise_bilateral(bright),
        "freq-0.25": denoise_frequency.denoise_frequency(bright, 0.25)}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (k, v) in zip(axes, outs.items()):
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(k)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: linear-gradient(135deg, #f197a7, #efb66f, #f9fc95); padding: 30px; border-radius: 26px; color: #3a2c2c; box-shadow: 0 8px 22px rgba(0,0,0,0.16); margin-top: 28px;">
  <h1 style="color: #3a2c2c; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">🧹 Step 5: Morphological cleanup</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #3a2c2c;">
    Hand-defined 3x3 opening/closing after aggressive brightening. 5-sample result: opening drops
    PSNR 22.3 → 19.5 — at this kernel scale it erases detail the metrics reward, so it stays OUT
    of the frozen pipeline unless full-val numbers disagree.
  </p>
</div>

<div style="background: linear-gradient(135deg, #7dbcf3, #9ff98d, #d6a870); padding: 30px; border-radius: 26px; text-align: center; color: #1d3557; box-shadow: 0 8px 20px rgba(0,0,0,0.15); margin-top: 28px;">
  <h1 style="font-size: 36px; margin-bottom: 8px;">🏁 Frozen pipeline</h1>
  <p style="font-size: 18px;">One ordered sequence, applied uniformly to LoLI-Street val and ExDark.</p>
</div>

### Candidate comparison (full LoLI-Street val, 3000 pairs)

Run: `<venv>/bin/python src/classical/pipeline.py --out results/classical.json`

| Candidate | MSE | PSNR | SSIM |
|---|---|---|---|
| raw | 8876 | 9.68 | 0.545 |
| equalize | 869 | 19.95 | 0.798 |
| equalize_freq | 882 | 19.78 | 0.790 |
| gamma_freq | 1760 | 16.05 | 0.745 |
| ssr_freq | 3525 | 12.87 | 0.709 |
| clahe_gamma_bilateral | 1256 | 17.37 | 0.829 |

PSNR and SSIM disagree (equalize vs clahe_gamma_bilateral), so the detector voted:
zero-shot YOLOv8n mAP@0.5 on 300 val frames is **0.725 (clahe_gamma_bilateral)**
vs 0.690 (equalize). Frozen pipeline: **clahe_gamma_bilateral** — best SSIM and best
task score, with the PSNR tradeoff (17.37 vs 19.95) stated plainly.